# SemIf · Training

<a href="https://colab.research.google.com/github/furbyhaxx/semlf/blob/master/notebooks/SemIf_Train.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

Fine-tune a LoRA adapter for decision-native readout. Each example is the exact `direct-options-v1` scoring prompt, and the loss covers only the gold answer letter, so training optimizes the same last-position logits that inference reads. Use it to recover accuracy lost to 4-bit weights, to adapt a smaller base model, or to fit your own decision data.

To run this, press *Runtime* → *Run all* on a **free** Tesla T4 Colab instance.
Companion notebooks: [inference](https://colab.research.google.com/github/furbyhaxx/semlf/blob/master/notebooks/SemIf_Inference.ipynb) · [evaluation](https://colab.research.google.com/github/furbyhaxx/semlf/blob/master/notebooks/SemIf_Eval.ipynb) · [training](https://colab.research.google.com/github/furbyhaxx/semlf/blob/master/notebooks/SemIf_Train.ipynb).

### Installation

Same stack as Unsloth's Qwen3.5 (4B) notebook: Unsloth, its bundled gated-delta-net
kernels and `causal_conv1d` for the Qwen3.5 linear-attention layers. No vLLM:
every decision is one forward pass, so there is nothing to serve.

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
# Unsloth bundles the gated delta net kernels; a leftover pip fla would shadow them
!uv pip uninstall -qqq flash-linear-attention fla-core
# causal_conv1d is supported only on torch==2.8.0. If you have newer torch versions, please wait 10 minutes!
!uv pip install --no-build-isolation causal_conv1d==1.6.0
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

In [ ]:
# @title Fetch SemIf { display-mode: "form" }
# SemIf is put on sys.path instead of pip-installed: its pyproject pins torch 2.10,
# which would replace the torch Unsloth was installed against.
import json, subprocess, sys, time
from pathlib import Path

SEMLF_REPO = "https://github.com/furbyhaxx/semlf"  # @param {type:"string"}
SEMLF_REF = "master"  # @param {type:"string"}

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "semif_phase1").is_dir()), None)
if ROOT is None:
    ROOT = (Path("/content") if Path("/content").is_dir() else Path.cwd()) / "semlf"
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", SEMLF_REF, SEMLF_REPO, str(ROOT)], check=True)
for path in (ROOT / "src", ROOT / "notebooks"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
OUT = ROOT / "outputs"
OUT.mkdir(exist_ok=True)
COMMIT = subprocess.run(["git", "-C", str(ROOT), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print(f"SemIf at {ROOT} ({COMMIT[:12] or 'not a git checkout'}); outputs go to {OUT}")

### Configuration

Pick a pinned model from the list or type any Hub ID, local checkpoint, or a LoRA adapter directory saved by the training notebook.

In [ ]:
# @title Model { display-mode: "form" }
MODELS = {  # pinned revisions; any other Hub model or local path also works
    "Qwen/Qwen3.5-4B": "851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a",  # reference: committed BF16 predictions
    "Qwen/Qwen3.5-2B": "15852e8c16360a2fea060d615a32b45270f8a8fc",
    "Qwen/Qwen3.5-0.8B": "2fc06364715b967f1860aea9cf38778875588b17",
    "Qwen/Qwen3.5-9B": "c202236235762e1c871ad0ccb60c8ee5ba337b9a",
    "Qwen/Qwen3-4B-Instruct-2507": "cdbee75f17c01a7cc42f958dc650907174af0554",
    "Qwen/Qwen3-1.7B": "70d244cc86ccca08cf5af4e1e306ecf908b1ad5e",
    "Qwen/Qwen3-0.6B": "c1899de289a04d12100db370d81485cdf75e47ca",
    "openbmb/MiniCPM5-2B": "12a3808a956f869c767195e9266b59c4d21d92e2",
}

MODEL = "Qwen/Qwen3.5-4B"  # @param ["Qwen/Qwen3.5-4B", "Qwen/Qwen3.5-2B", "Qwen/Qwen3.5-0.8B", "Qwen/Qwen3.5-9B", "Qwen/Qwen3-4B-Instruct-2507", "Qwen/Qwen3-1.7B", "Qwen/Qwen3-0.6B", "openbmb/MiniCPM5-2B"] {allow-input: true}
REVISION = ""  # @param {type:"string"}
WEIGHTS = "auto"  # @param ["auto", "16bit", "bnb-8bit", "bnb-4bit"]
MAX_SEQ_LENGTH = 4096  # @param {type:"integer"}
LORA_R = 16  # @param {type:"integer"}
LORA_ALPHA = 16  # @param {type:"integer"}
WANLI_PER_LABEL = 500  # @param {type:"integer"}
OPTION_COPIES = 2  # @param {type:"integer"}
EXTRA_TRAIN_JSONL = ""  # @param {type:"string"}
MAX_TRAIN_TOKENS = 1024  # @param {type:"integer"}
MAX_STEPS = 150  # @param {type:"integer"}
EPOCHS = 1  # @param {type:"number"}
LEARNING_RATE = 1e-4  # @param {type:"number"}
PER_DEVICE_BATCH = 4  # @param {type:"integer"}
GRAD_ACCUM = 4  # @param {type:"integer"}
BATCH_SIZE = 16  # @param {type:"integer"}
EVAL_WANLI256 = False  # @param {type:"boolean"}
ADAPTER_NAME = ""  # @param {type:"string"}

### GPU and precision

`WEIGHTS = "auto"` keeps 16-bit weights whenever they fit, because that preserves
the reference accuracy, and drops to 4-bit NF4 only when they do not. Choose it
yourself to trade accuracy for memory:

| Card | VRAM | Qwen3.5-4B suggestion |
| --- | --- | --- |
| A100 / L4 / RTX 3090-4090 | 22-80 GB | `16bit` in bf16, the reference setting |
| T4 / V100 / RTX 2080 Ti | 11-16 GB | `16bit` in fp16 (no bf16 before Ampere) |
| RTX 2060-3070, 8 GB | 6-8 GB | `bnb-4bit`, or a smaller base model |

`bnb-8bit` is the most faithful quantized format but is usually *slower* than
16-bit; `bnb-4bit` is smaller and faster than 8-bit. Unsloth's Triton kernels
need compute capability 7.0 or newer (V100, T4, RTX 20xx and later).

In [ ]:
import unsloth  # import before transformers so Unsloth can patch it
import torch
from huggingface_hub import model_info
from semif_phase1 import unsloth_backend as ub

GPU = ub.gpu_profile()
print(f"{GPU['name']}: {GPU['total_gib']} GiB, compute {GPU['compute_capability']}, bf16={GPU['bf16']}")
if float(GPU["compute_capability"]) < 7.0:
    print("WARNING: below compute 7.0; Unsloth's Triton kernels are unsupported on this card.")

revision = REVISION or MODELS.get(MODEL)
WEIGHTS_USED = WEIGHTS
if WEIGHTS == "auto":
    if Path(MODEL).exists():
        WEIGHTS_USED = "16bit"
    else:
        info = model_info(MODEL, revision=revision)
        params = info.safetensors.total if info.safetensors else None
        WEIGHTS_USED = ub.suggest_weights(params, GPU["total_bytes"]) if params else "16bit"
        print(f"{MODEL}: {params / 1e9:.2f}B parameters" if params else f"{MODEL}: parameter count unknown")
print(f"weights: {WEIGHTS_USED}")

### Load through Unsloth

`16bit` gives 16-bit LoRA, which fits Qwen3.5-4B on a T4 (Unsloth's own Qwen3.5 notebook does the same); `bnb-4bit` gives QLoRA for 8 GB cards. Evaluate the adapter in the precision you will serve it in.

In [ ]:
model, tokenizer, META = ub.load_model(
    MODEL,
    revision,
    load_in_4bit = WEIGHTS_USED == "bnb-4bit",
    load_in_8bit = WEIGHTS_USED == "bnb-8bit",
    max_seq_length = MAX_SEQ_LENGTH,
    for_training = True,
    use_gradient_checkpointing = "unsloth",
)
META["semlf_commit"] = COMMIT
from unsloth import FastLanguageModel, FastVisionModel
LOADER = FastVisionModel if META["loader"] == "FastVisionModel" else FastLanguageModel
print(json.dumps(META, indent = 1))

### Data

By default the training set is a class-balanced sample of the WANLI **train** split, fetched from the same pinned revision as the WANLI test rows and SHA-256 checked. Any WANLI seed that produced a frozen test row is dropped. Each row is repeated `OPTION_COPIES` times with shuffled option order, so the letters carry no answer prior.

Add your own rows with `EXTRA_TRAIN_JSONL`: the `examples/decisions.jsonl` format plus an integer `label` indexing the gold option. Evaluation suites are held out and checked for shared IDs, groups and text.

In [ ]:
import colab_helpers as ch
from semif_phase1 import training

wanli_path = ch.download_verified(training.WANLI_TRAIN_URL, training.WANLI_TRAIN_SHA256,
                                  OUT / "sources" / "wanli-train.jsonl")
train_rows = []
if WANLI_PER_LABEL > 0:
    train_rows += training.wanli_training_rows(
        wanli_path, ROOT / "benchmarks" / "manifests" / "source-selection.jsonl", WANLI_PER_LABEL)
if EXTRA_TRAIN_JSONL:
    train_rows += ch.read_jsonl(EXTRA_TRAIN_JSONL)
train_rows = training.augment(train_rows, OPTION_COPIES)

SUITES = ch.load_suites(["authored144", "perturbations108"] + (["wanli256"] if EVAL_WANLI256 else []), OUT, ROOT)
training.check_disjoint(train_rows, *SUITES.values())
print(f"{len(train_rows)} training rows; held out: { {name: len(rows) for name, rows in SUITES.items()} }")

In [ ]:
from datasets import Dataset

features, skipped = [], 0
for row in train_rows:
    try:
        features.append(training.answer_slot_features(tokenizer, row, MAX_TRAIN_TOKENS))
    except ValueError:
        skipped += 1
train_dataset = Dataset.from_list(features).shuffle(seed = 3407)
print(f"{len(train_dataset)} examples, {skipped} skipped for length")
example = train_dataset[0]
print(repr(tokenizer.decode(example["input_ids"][-40:])))
print("supervised token:", repr(tokenizer.decode([t for t in example["labels"] if t != training.IGNORE_INDEX])))

### Baseline

Score the held-out suites before training, in the same precision, so the before/after comparison isolates the adapter.

In [ ]:
from tqdm.auto import tqdm
import pandas as pd
from semif_phase1.unsloth_backend import score_rows

def scorer(batch_size):
    return lambda rows, progress: score_rows(model, tokenizer, rows, META, mode = "batched",
                                             batch_size = batch_size, progress = progress)

LOADER.for_inference(model)
BEFORE, _ = ch.evaluate_suites(scorer(BATCH_SIZE), SUITES, True, ROOT, tqdm)
pd.DataFrame(ch.summary_rows(BEFORE, stage = "before"))

### LoRA adapters

Only language layers are adapted; the vision tower of Qwen3.5 stays frozen because training rows are text-only.

In [ ]:
if LOADER is FastVisionModel:
    model = FastVisionModel.get_peft_model(
        model,
        finetune_vision_layers = False,
        finetune_language_layers = True,
        finetune_attention_modules = True,
        finetune_mlp_modules = True,
        r = LORA_R,
        lora_alpha = LORA_ALPHA,
        lora_dropout = 0,
        bias = "none",
        random_state = 3407,
    )
else:
    model = FastLanguageModel.get_peft_model(
        model,
        r = LORA_R,
        lora_alpha = LORA_ALPHA,
        lora_dropout = 0,
        bias = "none",
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        use_gradient_checkpointing = "unsloth",
        random_state = 3407,
    )
model.print_trainable_parameters()

### Train

A plain `transformers.Trainer` with pre-tokenized features: the labels mask everything except the answer letter, and Unsloth's patched model supplies the fast LoRA kernels, gradient checkpointing and memory-light cross-entropy. `MAX_STEPS = 150` fits a free Colab session; set it to `-1` to train for `EPOCHS`.

In [ ]:
import math
from transformers import DataCollatorForSeq2Seq, Trainer, TrainingArguments
from unsloth import is_bf16_supported

steps = MAX_STEPS if MAX_STEPS > 0 else math.ceil(len(train_dataset) / (PER_DEVICE_BATCH * GRAD_ACCUM) * EPOCHS)
LOADER.for_training(model)
trainer = Trainer(
    model = model,
    train_dataset = train_dataset,
    data_collator = DataCollatorForSeq2Seq(tokenizer, padding = True, pad_to_multiple_of = 8,
                                           label_pad_token_id = training.IGNORE_INDEX),
    args = TrainingArguments(
        output_dir = str(OUT / "checkpoints"),
        per_device_train_batch_size = PER_DEVICE_BATCH,
        gradient_accumulation_steps = GRAD_ACCUM,
        warmup_steps = max(1, steps // 20),
        max_steps = MAX_STEPS,
        num_train_epochs = EPOCHS,
        learning_rate = LEARNING_RATE,
        lr_scheduler_type = "cosine",
        optim = "adamw_8bit",
        weight_decay = 0.01,
        logging_steps = 5,
        save_strategy = "no",
        seed = 3407,
        report_to = "none",
        fp16 = not is_bf16_supported(),
        bf16 = is_bf16_supported(),
        remove_unused_columns = False,
    ),
)

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
trainer_stats = trainer.train()

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
print(f"{trainer_stats.metrics['train_runtime']:.0f} seconds used for training.")
print(f"Peak reserved memory = {used_memory} GB ({used_memory_for_lora} GB for training).")
history = [entry["loss"] for entry in trainer.state.log_history if "loss" in entry]
print(f"loss first -> last: {history[0]:.4f} -> {history[-1]:.4f}" if len(history) > 1 else history)

### After training

The same suites, precision and batch size as the baseline. Keep the adapter only if `authored144` and `perturbations108` hold up: WANLI is a different distribution, and gains there can cost accuracy on your decisions.

In [ ]:
LOADER.for_inference(model)
AFTER, AFTER_PREDICTIONS = ch.evaluate_suites(scorer(BATCH_SIZE), SUITES, True, ROOT, tqdm)
pd.DataFrame(ch.summary_rows(BEFORE, stage = "before") + ch.summary_rows(AFTER, stage = "after"))

### Save the adapter

This saves only the LoRA weights plus `semif_training.json`, a manifest with the base revision, data, hyperparameters and before/after metrics. Point the inference or evaluation notebook's `MODEL` at the directory to use it.

In [ ]:
name = ADAPTER_NAME or f"lora-{MODEL.replace('/', '_')}-{WEIGHTS_USED}-{time.strftime('%Y%m%d-%H%M%S')}"
ADAPTER_DIR = OUT / name
if ADAPTER_DIR.exists():
    raise FileExistsError(f"{ADAPTER_DIR} exists; choose a new ADAPTER_NAME")
model.save_pretrained(str(ADAPTER_DIR))
getattr(tokenizer, "processor", tokenizer).save_pretrained(str(ADAPTER_DIR))
ch.write_create_only(ADAPTER_DIR / "semif_training.json", {
    "base_model": MODEL,
    "base_revision": META["revision"],
    "weights": WEIGHTS_USED,
    "semlf_commit": COMMIT,
    "gpu": GPU,
    "objective": "cross-entropy on the gold answer letter of the direct-options-v1 prompt",
    "data": {"wanli_train_sha256": training.WANLI_TRAIN_SHA256, "wanli_per_label": WANLI_PER_LABEL,
             "extra_train_jsonl": EXTRA_TRAIN_JSONL or None, "option_copies": OPTION_COPIES,
             "examples": len(train_dataset), "skipped_for_length": skipped},
    "hyperparameters": {"lora_r": LORA_R, "lora_alpha": LORA_ALPHA, "learning_rate": LEARNING_RATE,
                        "max_steps": MAX_STEPS, "epochs": EPOCHS, "per_device_batch": PER_DEVICE_BATCH,
                        "grad_accum": GRAD_ACCUM, "max_train_tokens": MAX_TRAIN_TOKENS},
    "train_runtime_seconds": trainer_stats.metrics["train_runtime"],
    "before": BEFORE,
    "after": AFTER,
})
print(f"saved {ADAPTER_DIR}")
# model.push_to_hub("your_name/semif_lora", token = "YOUR_HF_TOKEN")

### Merged 16-bit and GGUF exports

Optional. A merged 16-bit checkpoint loads without PEFT. GGUF targets llama.cpp and the browser demo's wllama runtime; Unsloth builds llama.cpp for the conversion, and quantized GGUF should be re-evaluated before use.

In [ ]:
if False: model.save_pretrained_merged(str(OUT / f"{name}-merged"), tokenizer)
if False: model.save_pretrained_gguf(str(OUT / f"{name}-gguf"), tokenizer, quantization_method = "q4_k_m")

### Notes

* Probabilities are conditional option scores over the supplied options, not
  calibrated confidences. Validate them on your own workload before acting on them.
* Outputs are create-only, matching the repository's benchmark policy: every run
  writes a new file under `outputs/`.
* The Unsloth install cell follows
  [unslothai/notebooks](https://github.com/unslothai/notebooks) (LGPL-3.0).